# Optimización y series temporales

Tasa de aprendizaje dinámica, optimización por error, detector de sobre-generalización, series sintéticas, clasificador caótico.

Parte del **laboratorio neuronal**. Cada sección es un experimento independiente: ejecuta solo la que te interese, de arriba hacia abajo (cada una instala lo que necesita).

#Series Temporales Sintéticas

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime, timedelta

# --- 1. Generación de Series Temporales Sintéticas ---

def generar_serie_simple(num_puntos=100, tendencia=0.5, ruido_escala=5, inicio_fecha=None):
    """
    Genera una serie temporal sintética con una tendencia lineal y ruido aleatorio.

    Args:
        num_puntos (int): Número de puntos en la serie.
        tendencia (float): Magnitud de la tendencia lineal por punto.
        ruido_escala (float): Escala del ruido aleatorio añadido.
        inicio_fecha (str, optional): Fecha de inicio en formato 'YYYY-MM-DD'. Si es None, usa índices numéricos.

    Returns:
        pd.DataFrame: DataFrame con las columnas 'Momento' y 'Valor'.
    """
    if inicio_fecha:
        base_fecha = datetime.strptime(inicio_fecha, '%Y-%m-%d')
        indices = [base_fecha + timedelta(days=i) for i in range(num_puntos)]
    else:
        indices = np.arange(num_puntos)

    valores = (np.arange(num_puntos) * tendencia) + (np.random.randn(num_puntos) * ruido_escala)

    df = pd.DataFrame({'Momento': indices, 'Valor': valores})
    return df

def generar_serie_estacional(num_dias=365, periodo=30, ruido_escala=0.5, inicio_fecha='2023-01-01', amplitud_estacional=10, tendencia_final_valor=5):
    """
    Genera una serie temporal sintética con un patrón estacional, tendencia y ruido.

    Args:
        num_dias (int): Número de días para la serie.
        periodo (int): Período del patrón estacional (e.g., 30 para ciclo mensual).
        ruido_escala (float): Escala del ruido aleatorio.
        inicio_fecha (str): Fecha de inicio en formato 'YYYY-MM-DD'.
        amplitud_estacional (float): Amplitud de la componente estacional.
        tendencia_final_valor (float): Valor final de la tendencia lineal.

    Returns:
        pd.DataFrame: DataFrame con 'Tiempo' (datetime) y 'Valor'.
    """
    base_fecha = datetime.strptime(inicio_fecha, '%Y-%m-%d')
    fechas = [base_fecha + timedelta(days=i) for i in range(num_dias)]

    estacionalidad = np.sin(np.arange(num_dias) * (2 * np.pi / periodo)) * amplitud_estacional
    tendencia = np.linspace(0, tendencia_final_valor, num_dias)
    ruido = np.random.randn(num_dias) * ruido_escala

    valores = estacionalidad + tendencia + ruido

    df = pd.DataFrame({'Tiempo': fechas, 'Valor': valores})
    return df

def generar_serie_ar(longitud, coeficientes_ar, intercepto=0, ruido_std=1, inicio_fecha='2023-01-01'):
    """
    Genera una serie temporal usando un proceso autoregresivo (AR).

    Args:
        longitud (int): Longitud de la serie.
        coeficientes_ar (list): Lista de coeficientes (ej. [0.5, -0.2] para AR(2)).
        intercepto (float): Término constante.
        ruido_std (float): Desviación estándar del ruido.
        inicio_fecha (str): Fecha de inicio en formato 'YYYY-MM-DD'.

    Returns:
        pd.Series: Serie de pandas con índices de tiempo.
    """
    p = len(coeficientes_ar)
    serie = [0.0] * longitud # Inicializar

    # Rellenar los primeros 'p' valores para inicializar el proceso AR
    for i in range(p):
        serie[i] = np.random.normal(0, ruido_std)

    for i in range(p, longitud):
        valor_pasado = 0
        for j in range(p):
            valor_pasado += coeficientes_ar[j] * serie[i - 1 - j]
        serie[i] = intercepto + valor_pasado + np.random.normal(0, ruido_std)

    return pd.Series(serie, index=pd.to_datetime([datetime.strptime(inicio_fecha, '%Y-%m-%d') + timedelta(days=i) for i in range(longitud)]))

def generar_serie_combinada(num_puntos=365, tendencia_valor=0.1, ruido_escala=1, periodo_estacional=30, amplitud_estacional=5, coeficientes_ar=[0.5], inicio_fecha='2023-01-01'):
    """
    Genera una serie temporal combinando tendencia, estacionalidad, ruido y un componente AR.

    Args:
        num_puntos (int): Número de puntos en la serie.
        tendencia_valor (float): Magnitud de la tendencia lineal por punto.
        ruido_escala (float): Escala del ruido aleatorio.
        periodo_estacional (int): Período del patrón estacional.
        amplitud_estacional (float): Amplitud de la componente estacional.
        coeficientes_ar (list): Coeficientes para el componente AR.
        inicio_fecha (str): Fecha de inicio en formato 'YYYY-MM-DD'.

    Returns:
        pd.DataFrame: DataFrame con 'Tiempo' y 'Valor'.
    """
    base_fecha = datetime.strptime(inicio_fecha, '%Y-%m-%d')
    fechas = [base_fecha + timedelta(days=i) for i in range(num_puntos)]
    tiempo_indices = np.arange(num_puntos)

    # Componente de Tendencia
    tendencia_componente = tiempo_indices * tendencia_valor

    # Componente Estacional
    estacionalidad_componente = np.sin(tiempo_indices * (2 * np.pi / periodo_estacional)) * amplitud_estacional

    # Componente de Ruido
    ruido_componente = np.random.randn(num_puntos) * ruido_escala

    # Componente AR (simulada)
    ar_componente = generar_serie_ar(num_puntos, coeficientes_ar=coeficientes_ar, ruido_std=ruido_escala / 2, inicio_fecha=inicio_fecha).values # Usamos el valor AR puro, el ruido ya está incluido

    # Combinar todos los componentes
    valores = tendencia_componente + estacionalidad_componente + ruido_componente + ar_componente

    df = pd.DataFrame({'Tiempo': fechas, 'Valor': valores})
    return df

# --- 2. Visualización de las Series ---

def plot_serie_temporal(df, x_col, y_col, titulo, x_label, y_label):
    """
    Grafica una serie temporal.

    Args:
        df (pd.DataFrame): DataFrame con los datos de la serie.
        x_col (str): Nombre de la columna del eje X (tiempo/momento).
        y_col (str): Nombre de la columna del eje Y (valor).
        titulo (str): Título del gráfico.
        x_label (str): Etiqueta del eje X.
        y_label (str): Etiqueta del eje Y.
    """
    plt.figure(figsize=(12, 6))
    plt.plot(df[x_col], df[y_col], color='skyblue', linewidth=1.5)
    plt.title(titulo, fontsize=16)
    plt.xlabel(x_label, fontsize=12)
    plt.ylabel(y_label, fontsize=12)
    plt.grid(True, linestyle='--', alpha=0.7)
    plt.tight_layout()
    plt.show()

# --- Ejemplos de Uso y Visualización ---

# Serie Simple
print("--- Generando y Visualizando Serie Simple ---")
serie_simple_df = generar_serie_simple(num_puntos=150, tendencia=0.8, ruido_escala=10, inicio_fecha='2024-01-01')
print(serie_simple_df.head())
plot_serie_temporal(serie_simple_df, 'Momento', 'Valor',
                    'Serie Temporal Sintética Simple', 'Fecha', 'Valor')
print("-" * 50 + "\n")

# Serie Estacional
print("--- Generando y Visualizando Serie Estacional ---")
serie_estacional_df = generar_serie_estacional(num_dias=200, periodo=60, ruido_escala=2, inicio_fecha='2023-01-01')
print(serie_estacional_df.head())
plot_serie_temporal(serie_estacional_df, 'Tiempo', 'Valor',
                    'Serie Temporal Sintética con Estacionalidad', 'Fecha', 'Valor')
print("-" * 50 + "\n")

# Serie AR(1)
print("--- Generando y Visualizando Serie AR(1) ---")
serie_ar1_df = generar_serie_ar(longitud=50, coeficientes_ar=[0.8], ruido_std=0.5, inicio_fecha='2025-01-01').to_frame(name='Valor')
serie_ar1_df = serie_ar1_df.reset_index().rename(columns={'index': 'Tiempo'}) # Convertir el índice a columna
print(serie_ar1_df.head())
plot_serie_temporal(serie_ar1_df, 'Tiempo', 'Valor',
                    'Serie Temporal Sintética AR(1)', 'Fecha', 'Valor')
print("-" * 50 + "\n")

# Serie Combinada
print("--- Generando y Visualizando Serie Combinada ---")
serie_combinada_df = generar_serie_combinada(
    num_puntos=365 * 2, # 2 años de datos
    tendencia_valor=0.05,
    ruido_escala=3,
    periodo_estacional=90, # Ciclo trimestral
    amplitud_estacional=8,
    coeficientes_ar=[0.6, -0.1], # AR(2)
    inicio_fecha='2024-01-01'
)
print(serie_combinada_df.head())
plot_serie_temporal(serie_combinada_df, 'Tiempo', 'Valor',
                    'Serie Temporal Sintética Combinada (Tendencia + Estacionalidad + Ruido + AR)', 'Fecha', 'Valor')
print("-" * 50 + "\n")

#weighted_vote

In [ ]:
import numpy as np
import joblib
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score
import warnings
warnings.filterwarnings("ignore")

# --- Datos sintéticos ---
X, y = make_classification(n_samples=600, n_features=2, n_informative=2, n_redundant=0,
                           n_clusters_per_class=1, random_state=42, class_sep=1.2)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# --- Modelos base ---
base_classifiers = {
    "LogisticRegression": LogisticRegression(random_state=42),
    "KNeighbors": KNeighborsClassifier(n_neighbors=7),
    "SVC_Linear": SVC(kernel='linear', probability=True, random_state=42),
    "DecisionTree": DecisionTreeClassifier(max_depth=4, random_state=42),
    "GaussianNB": GaussianNB()
}

# --- Validación cruzada para obtener pesos ---
kf = KFold(n_splits=5, shuffle=True, random_state=42)
scores = {name: cross_val_score(clf, X_train, y_train, cv=kf, scoring='accuracy').mean()
          for name, clf in base_classifiers.items()}
weights = {k: v / sum(scores.values()) for k, v in scores.items()}

# --- Reentrenamiento completo ---
trained_models = {}
for name, clf in base_classifiers.items():
    clf.fit(X_train, y_train)
    trained_models[name] = clf

# --- Guardar modelos y pesos ---
joblib.dump(trained_models, '/content/ensemble_models.pkl')
joblib.dump(weights, '/content/ensemble_weights.pkl')
joblib.dump((X_test, y_test), '/content/test_data.pkl')

print("✅ Modelos y pesos guardados.")

In [ ]:
import numpy as np
import joblib
from sklearn.metrics import accuracy_score, f1_score
from matplotlib.colors import ListedColormap
import matplotlib.pyplot as plt

# --- Cargar modelos, pesos y datos ---
# Seguridad: joblib/pickle puede ejecutar código; carga solo archivos que hayas creado tú.
models = joblib.load('/content/ensemble_models.pkl')
weights = joblib.load('/content/ensemble_weights.pkl')
X_test, y_test = joblib.load('/content/test_data.pkl')

def weighted_vote(X, models, weights, num_classes=2):
    proba_total = np.zeros((X.shape[0], num_classes))
    for name, model in models.items():
        if hasattr(model, "predict_proba"):
            proba = model.predict_proba(X)
            proba_total += proba * weights[name]
    return np.argmax(proba_total, axis=1)

# --- Evaluación ---
y_pred = weighted_vote(X_test, models, weights)
acc = accuracy_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred, average='weighted')

print(f"🎯 Accuracy ensamblado: {acc:.4f} | F1-Weighted: {f1:.4f}")

# --- Frontera de decisión (visual opcional) ---
x_min, x_max = X_test[:, 0].min() - 1, X_test[:, 0].max() + 1
y_min, y_max = X_test[:, 1].min() - 1, X_test[:, 1].max() + 1
xx, yy = np.meshgrid(np.arange(x_min, x_max, 0.05),
                     np.arange(y_min, y_max, 0.05))

Z = weighted_vote(np.c_[xx.ravel(), yy.ravel()], models, weights)
Z = Z.reshape(xx.shape)

plt.figure(figsize=(8, 6))
plt.contourf(xx, yy, Z, cmap=ListedColormap(['#FFAAAA', '#AAFFAA']), alpha=0.6)
plt.scatter(X_test[:, 0], X_test[:, 1], c=y_test, cmap=ListedColormap(['#FF0000', '#00FF00']), edgecolor='k')
plt.title("Fronteras de Decisión del Ensamblado")
plt.show()

#Chaotic20DClassifier

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from tqdm import tqdm
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

config = {
    "lr": 5e-4,
    "batch_size": 512,
    "epochs": 1000,
    "early_stop_patience": 50,
    "model_path": "chaotic_best_model.pt",
}

def make_chaotic_20D_data(n=20000, noise=0.8):
    X = np.random.rand(n, 20) * 10 - 5
    y = np.zeros(n, dtype=int)
    for i, p in enumerate(X):
        if (p[0] > 2 and p[3] < -1 and p[7] > 3) or \
           (p[1] < -3 and p[5] > 4) or \
           (abs(p[9]) < .5 and p[12] > 2 and p[18] < -2): y[i] = 0
        elif (p[2] > 1 and p[4] < -.5) or \
             (p[6] < -2 and p[8] > 1.5 and np.sum(p[10:13]) > 5): y[i] = 1
        elif (p[10] > 0 and p[11] < 0 and p[13] > 0 and p[14] < 0) or \
             (np.sin(p[15]) * np.cos(p[16]) > 0.5 and p[17] < 1): y[i] = 2
        else: y[i] = 3
        p[0] += np.random.exponential(noise * 0.1) * np.random.choice([-1, 1])
        p[1] += np.random.normal(0, noise * 0.05)
        p[5] += np.random.poisson(noise * 0.5) * np.random.choice([-1, 1])
        p[19] += np.random.uniform(-noise, noise)
    return X, y

class Chaotic20DClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(20, 1024), nn.BatchNorm1d(1024), nn.LeakyReLU(0.01), nn.Dropout(0.4),
            nn.Linear(1024, 512), nn.BatchNorm1d(512), nn.LeakyReLU(0.01), nn.Dropout(0.3),
            nn.Linear(512, 256), nn.BatchNorm1d(256), nn.LeakyReLU(0.01), nn.Dropout(0.2),
            nn.Linear(256, 128), nn.BatchNorm1d(128), nn.LeakyReLU(0.01),
            nn.Linear(128, 4)
        )

    def forward(self, x):
        return self.net(x)

def load_data(batch_size):
    X, y = make_chaotic_20D_data()
    X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)
    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_val = scaler.transform(X_val)

    train_ds = torch.utils.data.TensorDataset(
        torch.tensor(X_train, dtype=torch.float32),
        torch.tensor(y_train, dtype=torch.long)
    )
    val_ds = torch.utils.data.TensorDataset(
        torch.tensor(X_val, dtype=torch.float32),
        torch.tensor(y_val, dtype=torch.long)
    )

    train_loader = torch.utils.data.DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=4, pin_memory=True)
    val_loader = torch.utils.data.DataLoader(val_ds, batch_size=batch_size*2, shuffle=False, num_workers=4, pin_memory=True)

    return train_loader, val_loader

def train_loop(model, optimizer, criterion, train_loader, val_loader):
    from torch.cuda.amp import autocast, GradScaler
    scaler = GradScaler()
    best_loss = float('inf')
    patience = 0

    for epoch in range(config["epochs"]):
        model.train()
        total_loss = 0
        pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}")
        for xb, yb in pbar:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            with autocast(enabled=device.type=='cuda'):
                out = model(xb)
                loss = criterion(out, yb)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            total_loss += loss.item()
            pbar.set_postfix(loss=total_loss/(pbar.n+1))

        val_loss = 0
        model.eval()
        with torch.no_grad():
            for xb, yb in val_loader:
                xb, yb = xb.to(device), yb.to(device)
                with autocast(enabled=device.type=='cuda'):
                    out = model(xb)
                    loss = criterion(out, yb)
                val_loss += loss.item()
        val_loss /= len(val_loader)

        if val_loss < best_loss:
            best_loss = val_loss
            torch.save(model.state_dict(), config["model_path"])
            patience = 0
        else:
            patience += 1
            if patience >= config["early_stop_patience"]:
                print(f"Early stopping en epoch {epoch+1}")
                break

        print(f"Epoch {epoch+1}: val_loss={val_loss:.4f}, best_loss={best_loss:.4f}")

if __name__ == "__main__":
    train_loader, val_loader = load_data(config["batch_size"])
    model = Chaotic20DClassifier().to(device)
    optimizer = optim.Adam(model.parameters(), lr=config["lr"], weight_decay=1e-5)
    criterion = nn.CrossEntropyLoss()
    train_loop(model, optimizer, criterion, train_loader, val_loader)

In [ ]:
import torch
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report
import os


config = {"model_path": "chaotic_best_model.pt"}
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

if not os.path.exists(config["model_path"]):
    raise FileNotFoundError(f"Modelo no encontrado en {config['model_path']}")

X, y = make_chaotic_20D_data()
scaler = StandardScaler().fit(X)
X = scaler.transform(X)

X = torch.tensor(X, dtype=torch.float32).to(device)
y = torch.tensor(y, dtype=torch.long).to(device)

model = Chaotic20DClassifier().to(device)
model.load_state_dict(torch.load(config["model_path"], map_location=device, weights_only=True))
model.eval()

with torch.no_grad():
    # You can use autocast context manager for mixed precision training on CUDA
    # It's generally good practice to keep it enabled when available
    if device.type == "cuda":
        from torch.cuda.amp import autocast
        with autocast():
            preds = model(X).argmax(dim=1)
    else:
        preds = model(X).argmax(dim=1)


print(classification_report(y.cpu(), preds.cpu()))

#ProgramadorDeTasaDeAprendizajeDinamico

In [ ]:
#import numpy as np

def actualizar_learning_rate_scheduler(
    current_lr: float,
    metrica_actual: float,
    mejor_metrica_registrada: float,
    epocas_sin_mejora: int,
    paciencia_lr: int,
    factor_reduccion_lr: float,
    min_lr: float,
    es_mejor_mayor: bool = True # True si una métrica mayor es mejor (ej. precisión), False si menor es mejor (ej. error)
) -> tuple[float, int, float, bool]:
    """
    Ajusta la tasa de aprendizaje basándose en el rendimiento de una métrica de monitoreo.

    Args:
        current_lr (float): Tasa de aprendizaje actual.
        metrica_actual (float): Valor de la métrica en la época actual.
        mejor_metrica_registrada (float): El mejor valor de la métrica registrado hasta ahora.
        epocas_sin_mejora (int): Número de épocas consecutivas sin mejora en la métrica.
        paciencia_lr (int): Número de épocas sin mejora para reducir la tasa de aprendizaje.
        factor_reduccion_lr (float): Factor por el cual se reduce la tasa de aprendizaje (ej. 0.1).
        min_lr (float): Valor mínimo al que puede descender la tasa de aprendizaje.
        es_mejor_mayor (bool): Indica si un valor *mayor* de la métrica es mejor (True, ej. precisión)
                               o si un valor *menor* es mejor (False, ej. error).

    Returns:
        tuple[float, int, float, bool]:
            - La nueva tasa de aprendizaje.
            - El nuevo contador de épocas sin mejora.
            - El nuevo mejor valor de la métrica registrada.
            - Un booleano indicando si la métrica mejoró en esta época.
    """
    mejoro_en_esta_epoca = False

    # Determina si la métrica actual es mejor que la mejor registrada hasta ahora
    if (es_mejor_mayor and metrica_actual > mejor_metrica_registrada) or \
       (not es_mejor_mayor and metrica_actual < mejor_metrica_registrada):
        mejor_metrica_registrada = metrica_actual
        epocas_sin_mejora = 0
        mejoro_en_esta_epoca = True
    else:
        epocas_sin_mejora += 1 # Incrementa las épocas sin mejora

    # Reduce la tasa de aprendizaje si la paciencia se agota
    if epocas_sin_mejora >= paciencia_lr:
        new_lr = current_lr * factor_reduccion_lr
        if new_lr >= min_lr:
            current_lr = new_lr
            epocas_sin_mejora = 0 # Reinicia la paciencia
        else:
            current_lr = min_lr # La tasa de aprendizaje no baja del mínimo

    return current_lr, epocas_sin_mejora, mejor_metrica_registrada, mejoro_en_esta_epoca

#OptimizaciónAdaptativaBasadaEnError

In [ ]:
import numpy as np
from dataclasses import dataclass


def OptimizaciónAdaptativaBasadaEnError(
    red: 'RedNeuronalDC', # Usa string para forward reference si RedNeuronalDC no está definida arriba
    X_train: np.ndarray,
    y_train: np.ndarray,
    learning_rate: float,
    epocas_por_ciclo_base: int = 50,
    umbral_mejora_minima: float = 0.0001,
    umbral_error_convergencia: float = 0.02,
    max_epocas_acumuladas: int = 1000
) -> tuple['RedNeuronalDC', list[float]]:
    """
    Entrena una Red Neuronal de forma adaptativa, ajustando el número de épocas
    en ciclos basados en la mejora del error.

    Args:
        red (RedNeuronalDC): La instancia de la red neuronal a entrenar.
        X_train (np.ndarray): Datos de entrada para el entrenamiento.
        y_train (np.ndarray): Etiquetas objetivo para el entrenamiento.
        learning_rate (float): La tasa de aprendizaje para las actualizaciones de pesos.
        epocas_por_ciclo_base (int): Número de épocas a entrenar en cada ciclo de verificación.
        umbral_mejora_minima (float): Si el error no mejora al menos esta cantidad
                                       en un ciclo, se puede detener el entrenamiento.
        umbral_error_convergencia (float): Si el error promedio cae por debajo de este umbral,
                                         se considera que la red ha convergido y se detiene.
        max_epocas_acumuladas (int): El número máximo total de épocas a entrenar,
                                     independientemente de la mejora.

    Returns:
        tuple[RedNeuronalDC, list[float]]:
            - La red neuronal entrenada.
            - Una lista del historial de errores promedio por cada época.
    """
    print("\n--- Iniciando Proceso de Optimización Adaptativa Basada en Error ---")
    historial_errores_total = []
    epocas_completadas = 0
    error_promedio_anterior_ciclo = float('inf') # Para comparar la mejora entre ciclos

    while epocas_completadas < max_epocas_acumuladas:
        # Calcular épocas para el ciclo actual, respetando el máximo total
        epocas_para_este_ciclo = min(epocas_por_ciclo_base, max_epocas_acumuladas - epocas_completadas)

        if epocas_para_este_ciclo <= 0:
            print("No quedan épocas disponibles dentro del límite máximo. Deteniendo.")
            break

        print(f"\n--- Ciclo de Entrenamiento: Épocas {epocas_completadas + 1} a {epocas_completadas + epocas_para_este_ciclo} ---")

        for epoch_in_cycle in range(epocas_para_este_ciclo):
            epoch_global_actual = epocas_completadas + epoch_in_cycle + 1
            error_total_epoca = 0

            # Mezclar datos de entrenamiento en cada época para aleatoriedad
            indices = np.arange(X_train.shape[0])
            np.random.shuffle(indices)
            X_shuffled, y_shuffled = X_train[indices], y_train[indices]

            for i in range(X_train.shape[0]):
                entrada = X_shuffled[i]
                objetivo = y_shuffled[i]

                # Propagación Hacia Adelante (Forward Pass)
                prediccion_final, salidas_capa_oculta_a, salidas_capa_oculta_z, salidas_capa_salida_z = predecir_red_dc(red, entrada)
                prediccion_final_valor = prediccion_final[0]

                # Cálculo del Error Instantáneo
                error_salida = objetivo - prediccion_final_valor
                error_total_epoca += 0.5 * (error_salida ** 2)

                # Retropropagación (Backpropagation)
                delta_salida = error_salida * derivada_sigmoide_de_salida(prediccion_final_valor)
                error_propagado_a_oculta = delta_salida * red.capa_salida[0].pesos
                delta_oculta = error_propagado_a_oculta * derivada_sigmoide_de_salida(salidas_capa_oculta_a)

                # Actualización de Pesos y Sesgos de la Capa de Salida
                red.capa_salida[0].pesos += learning_rate * delta_salida * salidas_capa_oculta_a
                red.capa_salida[0].sesgo += learning_rate * delta_salida

                # Actualización de Pesos y Sesgos de la Capa Oculta
                for j, neurona_oculta in enumerate(red.capa_oculta):
                    neurona_oculta.pesos += learning_rate * delta_oculta[j] * entrada
                    neurona_oculta.sesgo += learning_rate * delta_oculta[j]

            # Error promedio de la época
            avg_error_this_epoch = error_total_epoca / X_train.shape[0]
            historial_errores_total.append(avg_error_this_epoch)

            # Imprimir progreso de la época
            if (epoch_in_cycle + 1) % (epocas_para_este_ciclo // 5 if epocas_para_este_ciclo // 5 > 0 else 1) == 0 or (epoch_in_cycle + 1) == epocas_para_este_ciclo:
                print(f"  Época Global {epoch_global_actual}/{max_epocas_acumuladas}, Error Promedio: {avg_error_this_epoch:.6f}")

        epocas_completadas += epocas_para_este_ciclo # Actualizar el conteo total de épocas

        # Verificación del error al final del ciclo para decisiones adaptativas
        if historial_errores_total:
            error_actual_promedio_ciclo = historial_errores_total[-1]
            print(f"Error promedio al final de {epocas_completadas} épocas acumuladas: {error_actual_promedio_ciclo:.6f}")

            # Criterio de parada temprana: Convergencia por error objetivo
            if error_actual_promedio_ciclo <= umbral_error_convergencia:
                print(f"Error promedio ({error_actual_promedio_ciclo:.6f}) ha alcanzado el umbral de convergencia ({umbral_error_convergencia}). Deteniendo entrenamiento.")
                break

            # Criterio de parada temprana: Poca mejora
            mejora_en_error = error_promedio_anterior_ciclo - error_actual_promedio_ciclo
            if mejora_en_error < umbral_mejora_minima and epocas_completadas > epocas_por_ciclo_base: # Asegurar que no es la primera verificación
                print(f"Poca mejora en el error ({mejora_en_error:.6f} < {umbral_mejora_minima}). Deteniendo entrenamiento.")
                break

            error_promedio_anterior_ciclo = error_actual_promedio_ciclo
        else: # Si no hay historial de errores (caso muy raro, solo si X_train está vacío)
            break

    print("--- Proceso de Optimización Adaptativa Finalizado ---")
    return red, historial_errores_total

#SistemaInmuneCognitivo

In [ ]:
class OvergeneralizationDetector:
    """Detecta respuestas que usan generalizaciones excesivas como 'todos', 'nadie', etc."""
    def detect(self, ai_output: str) -> dict:
        patterns = ["todos los", "nadie puede", "siempre son", "nunca hacen", "todos piensan"]
        if any(p in ai_output.lower() for p in patterns):
            return {"type": "overgeneralization", "severity": 0.6, "message": "Detectada generalización excesiva o absoluta."}
        return {"type": None}